<a href="https://colab.research.google.com/github/DivyamAwasthy/Paper-Insight/blob/main/paper_insight_stage1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers feedparser

import feedparser
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


def fetch_arxiv(query: str = "cat:cs.LG", max_results: int = 30):
    """Fetch (title, abstract) pairs from arXiv for a given search query."""
    url = (
        "http://export.arxiv.org/api/query?"
        f"search_query={query}&start=0&max_results={max_results}"
        "&sortBy=submittedDate&sortOrder=descending"
    )
    feed = feedparser.parse(url)
    papers = []
    for entry in feed.entries:
        papers.append({
            "title": entry.title.strip().replace("\n", " "),
            "abstract": entry.summary.strip().replace("\n", " "),
        })
    return papers


MODEL_NAME = "all-MiniLM-L6-v2"


def embed_texts(model, texts):
    return model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def search(model, query, corpus_embeddings, papers, top_k=5):
    query_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(query_emb, corpus_embeddings)[0]
    ranked = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in ranked:
        results.append({
            "score": float(sims[idx]),
            "title": papers[idx]["title"],
            "abstract": papers[idx]["abstract"],
        })
    return results

print("Functions loaded: fetch_arxiv, embed_texts, search")

Functions loaded: fetch_arxiv, embed_texts, search


In [ ]:
papers = fetch_arxiv(query="all:graph+neural+network", max_results=100)
print(f"got {len(papers)} papers")

model = SentenceTransformer(MODEL_NAME)
corpus_embeddings = embed_texts(model, [p["abstract"] for p in papers])
print("embeddings shape:", corpus_embeddings.shape)   # expect (100, 384)

got 100 papers


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

embeddings shape: (100, 384)


In [ ]:
hits = search(model, "graph neural networks for molecules", corpus_embeddings, papers, top_k=5)
for i, h in enumerate(hits, 1):
    print(f"\n[{i}] score={h['score']:.3f}")
    print("   ", h["title"])
    print("   ", h["abstract"][:200], "...")


[1] score=0.368
    Blended Chart Surfaces: A Seamless Explicit Representation for Smooth Surface Fitting
    A surface representation suitable for geometry processing should be compact and explicit, provide global smoothness guarantees, support a wide range of surface topologies, and offer reliable access to ...

[2] score=0.356
    Deep Reinforcement Learning for Minimum Zero-Forcing Sets
    This paper explores the problem of finding the minimum zero-forcing set on undirected graphs and proposes an adapted machine-learning framework to solve the problem. The minimum zero-forcing set probl ...

[3] score=0.355
    Spatial Disease Mapping and Disparity Detection Using Generative AI: An Amortized Bayesian Learning Framework
    We introduce an amortized Bayesian framework for spatial boundary detection that generalizes posterior inference across areal graphs with varying numbers of regions and diverse adjacency structures. T ...

[4] score=0.354
    INI-VPINN: A Variational Physics-In

In [ ]:
papers = fetch_arxiv(query="all:graph+neural+network", max_results=100)
print(f"got {len(papers)} papers")

model = SentenceTransformer(MODEL_NAME)
corpus_embeddings = embed_texts(model, [p["abstract"] for p in papers])
print("embeddings shape:", corpus_embeddings.shape)   # expect (30, 384)

got 100 papers


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

embeddings shape: (100, 384)
